# Who Wants to Be a PoliMillionaire? — NLP 2025/26 Group Assignment

## Group members
- Jiaxin Yang — jiaxin.yang@mail.polimi.it
- Runjie (Simone) Dai — runjie.dai@mail.polimi.it

## Video
- Presentation video (≤5 min): https://youtu.be/Ub6XI_UnmH4?is=sGyjGIwOUgvoBZf9

## Statement on coding assistants
We used a coding assistant (Claude Code) to help write, refactor and document parts of the implementation. The system design, the experiments, the prompt/RAG/routing strategies and all analysis are our own; the assignment was **not** handed to an LLM to complete.


# 03 · Live play — the REAL test  (the game API)

**Who Wants to Be a PoliMillionaire?** — here the actual game we play, not our dev set.

The `live`-mode counterpart of notebook 01, this is:
- **01 = OFFLINE** ('our own test'): the hand-crafted dev set, gold known, accuracy computed locally.
- **03 = LIVE** ('the real test'): the game server feeds the questions and grades them; `correct`
  only **after** submitting we learn (from `AnswerResult`).

Same pipeline, same `EvalRecord`/JSONL log — only `config.mode='live'` and a logged-in `GameClient` differ.
The single switch is `run_session(pipeline, config, game_client=...)` (D-015).

> ⚠️ **This plays a REAL game.** A leaderboard attempt it consumes and the **30s/question timer it starts**.
> The leaderboard resets ~1 week before the deadline → save serious runs for then. Run the play cell
> deliberately you should — not by a stray Shift+Enter.

> **GPU needed.** Runtime ▸ Change runtime type ▸ T4 GPU, select you must.

## 1 · Setup — clone the repo, add `millionaire_client` to the path, install deps
The same clone as notebook 01, plus the course's `millionaire_client` package onto `sys.path` we add
(in the repo at `NLP_assignment_api_client/` it lives).

In [1]:
# Auto-reload edited src modules on every cell run -- so after a `git pull` the newest code lands without a
# manual importlib.reload or a restart. (Re-run the cell that USES the code, e.g. code-wire.)
# Colab's IPython ships an autoreload that does `from imp import reload`, and `imp` is GONE in Python 3.12 --
# so a tiny `imp` shim (reload only) we install first, then load the extension. BEST-EFFORT: any failure
# caught, so the cell never stalls (fall back: after a src pull, Runtime > Restart to pick changes up).
try:
    import sys as _sys, types as _types, importlib as _importlib
    if 'imp' not in _sys.modules:
        _imp = _types.ModuleType('imp')
        _imp.reload = _importlib.reload          # the one thing the old autoreload.py wants from `imp`.
        _sys.modules['imp'] = _imp
    _ip = get_ipython()
    _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')
    print('autoreload: ON (src edits hot-reload on cell re-run)')
except Exception as _e:
    print(f'autoreload OFF ({type(_e).__name__}: {_e}) -- after a src pull, Runtime > Restart to pick changes up.')

import os, sys

REPO_URL = 'https://github.com/SleepyEveryD/NLPDelivery.git'
REPO_ROOT = '/content/NLP'
BRANCH = 'main'
if not os.path.exists(REPO_ROOT):
  !git clone -b {BRANCH} {REPO_URL} {REPO_ROOT}
else:
  # Already cloned -> HARD-SYNC to the latest pushed 4-rag (fetch + force-reset to origin/{BRANCH}).
  # Re-run this cell and the newest src ALWAYS lands (the News/calculator fix etc.) -- no stale checkout,
  # no "local changes block the pull". Tracked files are overwritten to match remote; UNTRACKED run
  # outputs are KEPT (experiments/runs/* is gitignored; wrong_questions.jsonl is untracked).
  # NOTE: `git pull` updates the FILES on disk -- it does NOT refresh THIS notebook's cells in the Colab
  #       browser. Code changes under src/ apply on the next import; notebook-cell edits need a re-open.
  !cd {REPO_ROOT} && git fetch -q origin && git checkout -q -f -B {BRANCH} origin/{BRANCH}

# Confirm the branch + the EXACT synced commit (so "did my fix land?" you can verify at a glance).
!cd {REPO_ROOT} && echo "on branch:" $(git rev-parse --abbrev-ref HEAD) "@" $(git --no-pager log -1 --oneline)
!grep -q "def default_tools" {REPO_ROOT}/src/tools/__init__.py && echo "✅ default_tools present (phase-3 src)" || echo "❌ still old src"

# Our src tree AND the provided client package, onto the import path both go.
SRC = os.path.join(REPO_ROOT, 'src')
API_CLIENT = os.path.join(REPO_ROOT, 'NLP_assignment_api_client')
for p in (SRC, API_CLIENT):
  if p not in sys.path:
    sys.path.insert(0, p)
# Into the repo root, change directory we do -- relative paths simpler they become.
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)
print('On sys.path:', SRC, '|', API_CLIENT)

# The provided client, importable it must be -- confirm we do, before any game touch.
from millionaire_client import MillionaireClient
print('millionaire_client, imported it is.')

# --- Persistent model cache on Google Drive ------------------------------------------------------
# The Qwen 7B weights (~15GB) download ONCE to Drive, then every notebook/session reuses them --
# no repeat downloads. Set BEFORE transformers is imported, this must be. BEST-EFFORT: outside Colab,
# or if the Drive prompt you decline, fall back to the ephemeral cache (re-downloads, but never errors).
try:
    from google.colab import drive as _drive
    if not os.path.isdir('/content/gdrive/MyDrive'):
        _drive.mount('/content/gdrive')
    _hf = '/content/gdrive/MyDrive/hf_cache'
    os.makedirs(_hf, exist_ok=True)
    os.environ['HF_HOME'] = _hf
    print('HF cache ->', _hf, '(model downloads once, reused after)')
except Exception as _e:
    print(f'Drive cache off ({type(_e).__name__}) -- ephemeral cache, the model re-downloads each session.')

autoreload: ON (src edits hot-reload on cell re-run)
Cloning into '/content/NLP'...
remote: Enumerating objects: 190, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 190 (delta 84), reused 143 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (190/190), 1.25 MiB | 4.90 MiB/s, done.
Resolving deltas: 100% (84/84), done.
on branch: main @ 0ffcf91 .
✅ default_tools present (phase-3 src)
Repo root: /content/NLP
On sys.path: /content/NLP/src | /content/NLP/NLP_assignment_api_client
millionaire_client, imported it is.
Mounted at /content/gdrive
HF cache -> /content/gdrive/MyDrive/hf_cache (model downloads once, reused after)


In [2]:
# The inference stack + the client's `requests`, install we do (light it stays).
# NOTE: `-U` we MUST keep -- Colab a stale bitsandbytes (<0.46.1) preinstalls, and a bare
# `>=0.43.0` pip leaves it be (already satisfied), so the 4-bit loader then ImportErrors.
# `>=0.46.1` pinned + `-U` => the upgrade actually happens.
#
# ⚠️ pandas / requests, PINNED to Colab's own versions they are -- bare names + `-U` grabbed
# pandas 3.0.x & requests 2.34, which break `google-colab` (==2.2.2 / ==2.32.4) AND cudf / gradio /
# bqplot / db-dtypes (all need pandas<3). Matching Colab exactly -> zero dependency conflicts, and the
# pandas-3 breaking changes (copy-on-write, dropped APIs) never touch our metrics tables.
!pip install -q -U 'transformers>=4.45.0' 'accelerate>=0.34.0' 'bitsandbytes>=0.46.1' sentencepiece einops pyyaml 'pandas==2.2.2' matplotlib 'requests==2.32.4'
print('Installed, the dependencies are.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 129.0 MB/s eta 0:00:00
Installed, the dependencies are.


In [3]:
# Headless Chromium -- the live-NEWS body fetch it powers (configs/live.yaml: news_body_mode "browser").
# WHY a browser: the post-cutoff answer ("four Canadians quarantining..", a quote's speaker) lives in the
# article BODY, but the Google-News link is a JS consent-wall redirect `requests` cannot pass, and
# DuckDuckGo is blocked on the Colab IP. A real browser RUNS the JS -> past the wall, onto the publisher
# article -> the rendered text we read (RAW content, no synthesis; "headless Chromium" name in the video).
# Playwright + its Chromium + the system libs, install we do. Skip this you may if news_body_mode is "off".
!pip install -q playwright
!playwright install chromium
!playwright install-deps          # the system libs Chromium needs (Colab has sudo for this).

# ARMED? a REAL launch the surest test is. NOT ready -> News HEADLINES-only it falls back to (the body
# fetch crash-safe is, so the game never it blocks; set retrieval.news_body_mode: "off" to silence it).
try:
    from playwright.sync_api import sync_playwright
    with sync_playwright() as _p:
        _b = _p.chromium.launch(headless=True); _b.close()
    print('headless Chromium: READY -- live-News body fetch armed.')
except Exception as _e:
    print(f'headless Chromium NOT ready ({type(_e).__name__}: {_e})')
    print('   -> News will use HEADLINES only. Re-run this cell, or set retrieval.news_body_mode: "off".')


[autoreload of matplotlib.path failed: Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/extensions/autoreload.py", line 245, in check
    superreload(m, reload, self.old_objects)
  File "/usr/local/lib/python3.12/dist-packages/IPython/extensions/autoreload.py", line 410, in superreload
    update_generic(old_obj, new_obj)
  File "/usr/local/lib/python3.12/dist-packages/IPython/extensions/autoreload.py", line 347, in update_generic
    update(a, b)
  File "/usr/local/lib/python3.12/dist-packages/IPython/extensions/autoreload.py", line 302, in update_class
    if update_generic(old_obj, new_obj): continue
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/IPython/extensions/autoreload.py", line 347, in update_generic
    update(a, b)
  File "/usr/local/lib/python3.12/dist-packages/IPython/extensions/autoreload.py", line 266, in update_function
    setattr(old, name, getattr(new, name))
ValueError: __deepcopy__

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 21.9 MB/s eta 0:00:00
175.4 MiB [] 0% 0.0s175.4 MiB [] 0% 15.7s175.4 MiB [] 0% 10.5s175.4 MiB [] 1% 4.9s175.4 MiB [] 1% 3.7s175.4 MiB [] 2% 4.5s175.4 MiB [] 2% 4.4s175.4 MiB [] 2% 4.7s175.4 MiB [] 3% 3.8s175.4 MiB [] 4% 3.2s175.4 MiB [] 6% 2.8s175.4 MiB [] 7% 2.5s175.4 MiB [] 7% 2.7s175.4 MiB [] 8% 2.7s175.4 MiB [] 8% 2.9s175.4 MiB [] 9% 2.7s175.4 MiB [] 10% 2.5s175.4 MiB [] 11% 2.3s175.4 MiB [] 12% 2.2s175.4 MiB [] 13% 2.1s175.4 MiB [] 14% 2.1s175.4 MiB [] 15% 2.0s175.4 MiB [] 17% 1.9s175.4 MiB [] 18% 1.9s175.4 MiB [] 19% 1.8s175.4 MiB [] 20% 1.7s175.4 MiB [] 21% 1.8s175.4 MiB [] 21% 1.9s175.4 MiB [] 22% 1.8s175.4 MiB [] 24% 1.7s175.4 MiB [] 25% 1.6s175.4 MiB [] 27% 1.5s175.4 MiB [] 29% 1.4s175.4 MiB [] 30% 1.4s175.4 MiB [] 32% 1.3s175.4 MiB [] 34% 1.2s175.4 MiB [] 35% 1.2s175.4 MiB [] 37% 1.1s175.4 MiB [] 39% 1.1s175.4 MiB [] 40% 1.1s175.4 MiB [] 41% 1.1s175.4 MiB [] 43% 1.0s175.4 MiB [] 44% 1.0s175.4 MiB [] 46% 1.0s175.4 MiB [

In [4]:
import torch

# A GPU, present it must be -- on CPU, a 7B model in 30s answer we cannot.
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING -- no GPU. Runtime ▸ Change runtime type ▸ T4 GPU, switch you must.')

CUDA available: True
GPU: Tesla T4


## 2 · Load the LIVE run config
`configs/live.yaml` carries `mode: 'live'`. The competition + game mode, here you choose.

Competitions: 0 Entertainment · 1 History · 2 Science · 3 Maths · 4 Philosophy · 5 News.

In [5]:
from config import RunConfig

# The live config, from YAML we load.
config = RunConfig.from_yaml(os.path.join(REPO_ROOT, 'configs', 'live.yaml'))

# Which competition to play + how, here choose it you do.
config.game.competition_id = 0          # 0..5
config.game.game_mode = 'text'          # 'text' | 'speech'
config.run_id = f'live_comp{config.game.competition_id}'

# The Guardian Open Platform key -- from a Colab secret read it we do (B3: NEVER hardcoded). With it, the
# News body comes FIRST from the Guardian API (raw bodyText in ONE ~0.2s call -- no browser, no consent
# wall), and only NON-Guardian stories fall back to the headless browser. ABSENT -> the Guardian fast-path
# simply skips, the browser handles News (just slower). A FREE key: open-platform.theguardian.com/access
try:
    from google.colab import userdata as _ud
    config.retrieval.guardian_api_key = _ud.get('guardian_key') or ''
except Exception:
    config.retrieval.guardian_api_key = config.retrieval.guardian_api_key or ''

print('mode:', config.mode)
print('competition_id:', config.game.competition_id, '| game_mode:', config.game.game_mode)
print('aim_seconds:', config.game.aim_seconds, '(below the 30s wall, a network margin this keeps)')
print('model:', config.model.name, '|', config.model.quantization)
print('Guardian API:', 'KEY SET (fast body path armed)' if config.retrieval.guardian_api_key else 'no key -> News uses browser fallback')

mode: live
competition_id: 0 | game_mode: text
aim_seconds: 25.0 (below the 30s wall, a network margin this keeps)
model: Qwen/Qwen2.5-7B-Instruct | 4bit
Guardian API: KEY SET (fast body path armed)


## 3 · Load + warm up the model
Identical to notebook 01 — the cold-start cost paid **before** any timed question, so the 30s wall a cold load never eats.

In [6]:
import time
from inference.engine import TransformersEngine

# Once, the model we load -- the cold-start cost, here we pay it.
t0 = time.perf_counter()
if 'engine' not in globals():
      engine = TransformersEngine(model_name=config.model.name,
                                  quantization=config.model.quantization,
                                  dtype=config.model.dtype)
      engine.warmup()
else:
      print('engine already in VRAM, skipping load.')
print(f'Model loaded in {time.perf_counter() - t0:.1f}s')

# Warm up -- the first-call kernels, compiled before any timed question they are.
t0 = time.perf_counter()
engine.warmup()
print(f'Warmup in {time.perf_counter() - t0:.1f}s')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded in 232.8s
Warmup in 0.5s


## 4 · Wire the pipeline (same as offline) + log in to the game
DI exactly as notebook 01 (D-006) — only now a logged-in `GameClient` we also build.

Credentials: a PoliMi email as the username; the password from a **Colab secret** named
`poli-millionaire` we read (hardcode it, we must NOT — B3). Add it via the 🔑 panel on the left.

In [7]:
from classify.classifier import QuestionClassifier
from prompting.builder import PromptBuilder, RoutingPromptBuilder
from classify.reasoning_router import MATHS_LIVE_POLICY, MATHS_LIVE_FALLBACK
from agent.pipeline import QAPipeline
from tools import default_tools, solve_maths
from retrieval import build_retriever

# Phase 4 RAG: the routing retriever, built from config ONLY when config.retrieval.enabled.
# Ablation (RAG on vs off) = the `enabled` flag. `source` picks the strategy:
#   "routed"    -> per question: News -> live web (Google News RSS + headless-Chromium body), else -> FAISS/Wikipedia.
#   "wikipedia" | "web" | "faiss" -> single-backend ablations.
# `needs_retrieval` still gates per question. News (post-cutoff) the live web NEEDS -- Wikipedia alone
# left News at 2/7, the breaking 2026 facts it cannot hold; Google News headlines + the article body we add.
retriever = build_retriever(config.retrieval)
print('config.retrieval:', config.retrieval)   # the LOADED flags -- if RAG is OFF, here enabled=False you see.
print('RAG:', (f'ON  source={config.retrieval.source}  top_k={config.retrieval.top_k}') if retriever else 'OFF')

# The collaborators, injected into the pipeline they are (D-006) -- identical to offline.
pipeline = QAPipeline(
    engine=engine,
    prompt_builder=PromptBuilder(strategy=config.prompt_strategy),
    classifier=QuestionClassifier(),
    retriever=retriever,       # Phase 4: RAW evidence; News->web, else->FAISS/Wikipedia; gated + graceful.
    tools=default_tools(),     # Phase 3 ON: the safe-AST calculator. ONLY on maths questions it fires
                               # (needs_calculator gates it) -- on Entertainment etc. a harmless no-op it is.
    latency_budget_s=config.latency_budget_s,
)

# --- Maths (comp 3): REVERTED to the 5bcf593 known-good config (Maths 9/10, reached level 9) ---
# After 5bcf593 we tried two "improvements" that both made Maths worse:
#   * 403b989 added the calculator as a match-gated verifier -- it helped on some Qs but introduced new
#     failure modes (e.g. Q6767 group-theory: calc emitted 4*12/(2*2)=12 mapping to wrong option).
#   * 5130bac switched to cot_maths_v1 with worked exemplars -- the exemplars anchored variable setup
#     (Q6777 chain now writes "Let s=speed, p=price") but the model still slipped on the algebra step
#     and answer-to-option mapping; net result was no better than cot_v2.
# Reverting to the cot_v2 + NO-calculator + single-pass config until we have evidence a change wins.
# cot_maths_v1 stays REGISTERED (no harm) but unused; the calculator stays in src but Maths skips it.
# ADAPTIVE ROUTING (offline experiment, notebook 04): a RoutingPromptBuilder picks the prompt per
# question. CONSERVATIVE policy -- re-route ONLY the shapes we have evidence for to
# structured_enumeration_cot (interval-counting fixed 0.4->1.0, incl. the clock-chime death that capped
# Maths at level 9; temporal + discrete also helped); EVERYTHING ELSE (arithmetic, logic, concept/stats)
# stays on the battle-tested cot_v2 via the fallback. Minimal regression risk, targets the documented loss.
pipeline_maths = QAPipeline(
    engine=engine,
    prompt_builder=RoutingPromptBuilder(policy=MATHS_LIVE_POLICY, fallback_strategy=MATHS_LIVE_FALLBACK),
    classifier=QuestionClassifier(),
    retriever=None,            # Maths: NO retrieval -- it only distracts here.
    tools=None,                # NO calculator -- at n=1 it would clobber the chain on numeric Qs (run #8).
    solver=solve_maths,        # DETERMINISTIC type-specific solver -- short-circuits the LLM on solvable
                               # types (finite-field roots, gcd, characteristic, sum/product, reflection,
                               # triangle, %); abstains on all else (0 regressions on the logs).
    latency_budget_s=config.latency_budget_s,
    max_new_tokens=450,        # 30s WALL guard. Raised 300->400: B3 keeps only SHORT
                               # time-interval questions in structured enumeration, so cot_v2's terse chains
                               # now finish in 5-10s (verified run: max 10.5s, mean 5.4s, zero turns >25s) --
                               # big headroom. 450 lets a legitimately long chain reach 'Answer:' rather than
                               # truncate. CAVEAT: ~16 tok/s => 450 tok ~= 28s, ABOVE the 25s aim -- only safe
                               # because chains rarely run that long now; dial back toward 400 if any Maths
                               # turn nears the wall. (latency_budget is advisory -- the token cap is the guard.)
    # self_consistency_n defaults to 1 -- SC dropped for Maths (run #8: it timed out, gave no benefit).
)
print('Maths pipeline (comp 3): ADAPTIVE (counting/temporal/enum -> structured_enumeration_cot; else -> cot_v2)'
      ' + 300 tokens (30s-wall safe) + single-pass (n=1) + NO retrieval + NO calculator')

# --- Per-race pipelines (D-006): EACH competition its OWN QAPipeline, so tuning ONE race never bleeds
# into another. The assignment ENCOURAGES per-topic strategies ("are certain models better at certain
# topics?") and rewards it on both the leaderboard and the investigation score -- and News must stay
# stable while we tune the rest. Today every race except Maths is the SAME few_shot_v1+RAG recipe, just
# as a separate instance -> behaviour identical now, independently tunable later (give a race its own
# strategy/retriever/tools and only that race changes).
#   * the heavy FAISS/Wikipedia retriever the four KNOWLEDGE races SHARE (one model load, not four);
#   * News gets its OWN web retriever instance, so its config can be tuned in isolation;
#   * Maths uses no retriever (pipeline_maths above).
knowledge_retriever = retriever                     # the routed retriever built above (FAISS/Wikipedia).
news_retriever = build_retriever(config.retrieval)  # News' OWN instance (web path; light, no model load).

def _make_race_pipeline(retr):
    """A fresh QAPipeline twin of the shared recipe -- few_shot_v1 + RAG + classifier + tools."""
    return QAPipeline(
        engine=engine,
        prompt_builder=PromptBuilder(strategy=config.prompt_strategy),
        classifier=QuestionClassifier(),
        retriever=retr,
        tools=default_tools(),
        latency_budget_s=config.latency_budget_s,
    )

# --- Entertainment (comp 0): its OWN pipeline, `few_shot_entertainment` it runs (D-ENT) ---
# Entertainment is single-fact pop-culture recall (film / music / TV / books / games / sport). The shared
# few_shot_v1 primes the FORMAT with generic exemplars (capital-of, photosynthesis, 6x7) -- right shape,
# wrong register. `few_shot_entertainment` swaps those for film/music/TV exemplars + a domain-aware
# instruction, so the prime matches the questions asked and the small model reaches for the RIGHT kind of
# fact. NO chain-of-thought (recall drifts under CoT); RAG stays ON (Wikipedia/FAISS covers entertainment
# well and grounds the harder later-level facts), gated by `needs_retrieval`; the calculator is a harmless
# no-op here. Isolated from the other races -- editing this never touches News/Maths.
def _make_entertainment_pipeline(retr):
    return QAPipeline(
        engine=engine,
        prompt_builder=PromptBuilder(strategy="few_shot_entertainment"),
        classifier=QuestionClassifier(),
        retriever=retr,            # Phase 4: FAISS/Wikipedia evidence, gated per question.
        tools=default_tools(),     # no-op on entertainment (needs_calculator never fires here).
        latency_budget_s=config.latency_budget_s,
    )

pipeline_entertainment = _make_entertainment_pipeline(knowledge_retriever)
print('Entertainment pipeline (comp 0): few_shot_entertainment + RAG (FAISS/Wikipedia, gated) + no CoT')

RACE_PIPELINES = {
    0: pipeline_entertainment,                     # Entertainment (few_shot_entertainment + RAG)
    1: _make_race_pipeline(knowledge_retriever),   # Ancient History & Politics
    2: _make_race_pipeline(knowledge_retriever),   # Science & Nature
    3: pipeline_maths,                             # Maths (cot_v2 routing, NO retrieval, NO tools)
    4: _make_race_pipeline(knowledge_retriever),   # Philosophy & Psychology
    5: _make_race_pipeline(news_retriever),        # News (its OWN web retriever)
}

def pipeline_for(cid):
    """The pipeline for competition `cid` -- each race its own, so per-race tuning stays isolated."""
    return RACE_PIPELINES.get(cid, pipeline)

print('Per-race pipelines wired:',
      {c: ('entertainment' if c == 0 else 'maths' if c == 3 else 'news' if c == 5 else 'knowledge') for c in range(6)})

# --- Log in to the real game ---
from google.colab import userdata
from game.client import GameClient

USERNAME = userdata.get('username')       # <-- your PoliMi email, fill it you must.
PASSWORD = userdata.get('password')  # <-- a Colab secret, NOT hardcoded (B3).

game_client = GameClient()
game_client.login(USERNAME, PASSWORD)
print('Logged in as', USERNAME)

# The competitions and their ids, list them we do (safe -- no timer this starts).
for c in game_client.list_competitions():
    ml = getattr(c, 'max_levels', '?')
    print('  id=', c.id, '|', c.name, '| max_levels=', ml)

config.retrieval: RetrievalConfig(enabled=True, source='routed', top_k=3, embedder='intfloat/multilingual-e5-small', index_path='data/corpus/simplewiki', bm25_index_path=None, min_score=0.7, news_fetch_bodies=3, news_body_mode='browser', guardian_api_key='87118f74-361a-4b36-be92-dedc7ca39839')
RAG: ON  source=routed  top_k=3
Maths pipeline (comp 3): ADAPTIVE (counting/temporal/enum -> structured_enumeration_cot; else -> cot_v2) + 300 tokens (30s-wall safe) + single-pass (n=1) + NO retrieval + NO calculator
Entertainment pipeline (comp 0): few_shot_entertainment + RAG (FAISS/Wikipedia, gated) + no CoT
Per-race pipelines wired: {0: 'entertainment', 1: 'knowledge', 2: 'knowledge', 3: 'maths', 4: 'knowledge', 5: 'news'}
Logged in as runjie dai
  id= 0 | Entertainment | max_levels= 15
  id= 1 | Ancient History and Politics | max_levels= 15
  id= 2 | Science and Nature | max_levels= 15
  id= 3 | Maths | max_levels= 15
  id= 4 | Philosophy and Psychology | max_levels= 15
  id= 5 | News | max_

### 🔬 Core implementation: multi-model / self-consistency voting + a safe calculator tool (Agentic AI)

The assignment encourages **agentic** techniques: tool calls + multi-model combinations. Our two core components, with source shown directly below.

**(a) Majority vote** (`src/agent/voting.py`) -- one primitive serves two callers: (1) self-consistency (N sampled CoT chains from the same model vote) and (2) Phase-5 ensembling (different models each cast one vote). Confidence is the **vote share**, a genuine calibration signal (2/3 votes = the uncertainty of "two agree, one dissents"):

```python
def majority_vote(predictions: list[Prediction]) -> Prediction:
    if not predictions:
        raise ValueError("majority_vote of an empty list ...")
    counts = Counter(p.answer for p in predictions)          # votes per answer
    top = max(counts.values())
    tied = [a for a, c in counts.items() if c == top]
    if len(tied) > 1:                                        # tie -> break by each member's mean confidence
        winner_answer = max(tied, key=lambda a: _mean_conf(a))
    else:
        winner_answer = tied[0]
    winners = [p for p in predictions if p.answer == winner_answer]
    rep = max(winners, key=lambda p: p.confidence)           # most confident winning sample as representative
    vote_share = top / len(predictions)                      # confidence = vote share (true calibration signal)
    return Prediction(qid=rep.qid, answer=winner_answer, confidence=vote_share,
                      tool_used=next((p.tool_used for p in winners if p.tool_used), None), ...)
```

**(b) Safe calculator tool** (`src/tools/calculator.py`) -- we **never** give the model the dangerous power of `eval()`; we only walk an arithmetic AST, safe by construction:

```python
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.FloorDiv: operator.floordiv, ast.USub: operator.neg, ast.UAdd: operator.pos}

def _eval(node):                       # allow numbers and arithmetic only, reject everything else
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.operand))
    raise ValueError("Forbidden, this expression is.")

def calculate(expression: str) -> float:           # "12 * (3 + 4) / 2" -> 42.0
    return _eval(ast.parse(expression, mode="eval").body)
```

In the pipeline the calculator is a **verifier** (see `_run_calculator_tool`): the model emits a JSON call, we compute the value, and **only when** it uniquely matches an option (±0.5%) do we override the original CoT answer -- avoiding mistakenly rewriting "an answer that is a conclusion using a number" (such as a t-test).

## 5 · ▶ The REAL test — sweep ALL competitions (live) + scores + every wrong question

Plays ONE live game in **each** of the 6 competitions (0–5), back to back, with a polite pause between
(the assignment asks: avoid rapid consecutive requests). Each competition logs its own run dir
(`live_comp{id}`). Then a per-competition **scoreboard** + the consolidated list of **every wrong
question** — text, options, and the letter our model picked — also saved to `experiments/wrong_questions.jsonl`.

> ⚠️ This plays **6 REAL games** (timers + leaderboard). Run it deliberately. **Maths (comp 3)** runs its
> own pipeline — `cot_v2` (CoT + option-matching check), **single-pass** (self-consistency was dropped after
> run #8 timed out at 41s), no retrieval, no calculator; every other competition uses the shared `few_shot_v1`
> pipeline. **RAG (Phase 4) is ON** when `configs/live.yaml` has `retrieval.enabled: true` —
> `source: "routed"` sends **News → live web (DuckDuckGo)** and every other topic → **FAISS corpus /
> Wikipedia**. (`needs_retrieval` still gates it per question.) One game failing (e.g. a rate-limit) won't
> abort the rest — it's logged as a blank row and the sweep continues.

In [8]:
from evaluation.runner import run_all_competitions

# Clear THIS sweep's prior run dirs FIRST. The logger appends within a run and the dir name is reused
# (live_comp{id}), so without this a re-run piles onto the previous one -> duplicate qids, inflated counts,
# mixed strategies. This is a SHELL command (not cached Python), so it works even if the logger's
# truncate-on-open fix hasn't loaded yet (that needs a kernel Restart; this rm does not).
!rm -rf {REPO_ROOT}/experiments/runs/live_comp*
print('cleared prior live_comp* run dirs')

COMP_NAMES = {0: 'Entertainment', 1: 'Ancient History & Politics', 2: 'Science & Nature',
              3: 'Maths', 4: 'Philosophy & Psychology', 5: 'News'}

# One live game in EVERY competition (0-5), back to back, a polite pause between (PDF: no rapid requests).
# Each competition its own run dir gets (live_comp{id}); one game's failure the rest never sinks.
# Maths (comp 3) its OWN pipeline gets via `pipeline_for` -- cot_v2 + single-pass (n=1) + NO retrieval + NO
# calculator (run #8: cot_v2 + self-consistency timed out at 41s on a question it answered CORRECTLY; SC gave
# no benefit, so dropped). Routed by competition_id, the reliable LIVE signal; every other competition the
# shared few_shot + RAG `pipeline` keeps.
comp_runs = run_all_competitions(
    pipeline, config, game_client,
    competition_ids=range(6),
    log_root=os.path.join(REPO_ROOT, 'experiments', 'runs'),
    pause_s=8.0,
    on_competition=lambda cid: print(f"\n===== ▶ Competition {cid}: {COMP_NAMES.get(cid, '?')} ====="),
    pipeline_for=pipeline_for,   # per-race: each competition its own pipeline (Maths/News/knowledge)
)
print('\nSweep done. Run paths:')
for cid, path in comp_runs:
    print(f"  comp {cid} {COMP_NAMES.get(cid, ''):28} -> {path}")

cleared prior live_comp* run dirs

===== ▶ Competition 0: Entertainment =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=195 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=2.5s (left was 29.911367)
[2] qid=392 lvl=0 reached=None -> A | correct=True | timed_out=False | latency=6.5s (left was 29.906298)
[3] qid=265 lvl=0 reached=None -> C | correct=True | timed_out=False | latency=6.2s (left was 29.90629)
[4] qid=98 lvl=0 reached=None -> A | correct=True | timed_out=False | latency=2.0s (left was 29.912476)
[5] qid=456 lvl=0 reached=None -> A | correct=True | timed_out=False | latency=6.1s (left was 29.916133)
[6] qid=365 lvl=0 reached=5 -> D | correct=False | timed_out=False | latency=1.8s (left was 29.913157)

===== ▶ Competition 1: Ancient History & Politics =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=1217 lvl=0 reached=None -> A | correct=True | timed_out=False | latency=5.5s (left was 29.914464)
[2] qid=1019 lvl=0 reached=None -> C | correct=True | timed_out=False | latency=4.7s (left was 29.909931)
[3] qid=1007 lvl=0 reached=None -> A | correct=True | timed_out=False | latency=5.6s (left was 29.91513)
[4] qid=888 lvl=0 reached=None -> C | correct=True | timed_out=False | latency=9.3s (left was 29.913465)
[5] qid=852 lvl=0 reached=None -> A | correct=True | timed_out=False | latency=9.3s (left was 29.915664)
[6] qid=1270 lvl=0 reached=None -> A | correct=True | timed_out=False | latency=9.3s (left was 29.913825)
[7] qid=876 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=5.1s (left was 29.914717)
[8] qid=1001 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=5.5s (left was 29.914798)
[9] qid=993 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=4.8s (left was 29.915235)
[10] qid=966 lvl=0 reached=None -> A | correct=True

...

[1] qid=6907 lvl=0 reached=0 -> A | correct=False | timed_out=False | latency=7.4s (left was 29.912924)

===== ▶ Competition 4: Philosophy & Psychology =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=9647 lvl=0 reached=None -> D | correct=True | timed_out=False | latency=5.7s (left was 29.916481)
[2] qid=7660 lvl=0 reached=None -> C | correct=True | timed_out=False | latency=5.3s (left was 29.915767)
[3] qid=7472 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=1.6s (left was 29.911735)
[4] qid=9464 lvl=0 reached=None -> A | correct=True | timed_out=False | latency=1.5s (left was 29.912567)
[5] qid=7663 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=1.5s (left was 29.913993)
[6] qid=9569 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=1.3s (left was 29.914739)
[7] qid=10201 lvl=0 reached=None -> C | correct=True | timed_out=False | latency=9.5s (left was 29.914707)
[8] qid=9497 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=1.5s (left was 29.915657)
[9] qid=7077 lvl=0 reached=None -> A | correct=True | timed_out=False | latency=1.5s (left was 29.913806)
[10] qid=7100 lvl=0 reached=None -> B | corre

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=10759 lvl=0 reached=None -> D | correct=True | timed_out=False | latency=12.6s (left was 29.915127)
[2] qid=11403 lvl=0 reached=None -> A | correct=True | timed_out=False | latency=6.6s (left was 29.91371)
[3] qid=10998 lvl=0 reached=None -> A | correct=True | timed_out=False | latency=5.3s (left was 29.910266)
[4] qid=11641 lvl=0 reached=3 -> A | correct=False | timed_out=False | latency=17.5s (left was 29.91325)

Sweep done. Run paths:
  comp 0 Entertainment                -> /content/NLP/experiments/runs/live_comp0
  comp 1 Ancient History & Politics   -> /content/NLP/experiments/runs/live_comp1
  comp 2 Science & Nature             -> /content/NLP/experiments/runs/live_comp2
  comp 3 Maths                        -> /content/NLP/experiments/runs/live_comp3
  comp 4 Philosophy & Psychology      -> /content/NLP/experiments/runs/live_comp4
  comp 5 News                         -> /content/NLP/experiments/runs/live_comp5


In [9]:
import json
import pandas as pd
from evaluation.metrics import load_runs


def _retr_docs(r):
    """The retrieved doc ids as a clean list, defensively we read (NaN/None -> [])."""
    v = r.get('retrieved_doc_ids')
    return list(v) if isinstance(v, (list, tuple)) else []


def _run_reached(df):
    """The level THIS run CLIMBED to -- the server's `reached_level` telemetry (from AnswerResult),
    its max across the turns. NOT `level` (the per-turn rung the server sends as 0 -- the old bug that
    made this column always 0). Fallbacks: current_level, then level. None only if the server said nothing."""
    for col in ('reached_level', 'current_level', 'level'):
        if col in df.columns:
            s = df[col].dropna()
            if not s.empty:
                return int(s.max())
    return None


def _lb_entry(cid):
    """The AUTHORITATIVE leaderboard row for us in this competition (reached_level + score) -- the REAL
    scored metric (D-014). Crash-safe: None on any error (player not found, client shape differs, ...)."""
    try:
        me = game_client._client.user.username
        return game_client._client.leaderboard.find_player(cid, me)
    except Exception:
        return None


rows, wrong_all, usage = [], [], []
for cid, path in comp_runs:
    lb = _lb_entry(cid)
    lb_level = getattr(lb, 'reached_level', None)
    lb_score = getattr(lb, 'score', None)
    if path is None:   # That competition failed (e.g. rate-limited) -- a blank row it gets (lb still shown).
        rows.append({'comp': cid, 'name': COMP_NAMES.get(cid, ''), 'answered': 0,
                     'correct': 0, 'of': 0, 'accuracy': float('nan'),
                     'run_reached': None, 'lb_level': lb_level, 'lb_score': lb_score})
        continue
    df = load_runs([path])
    known = df[df['correct'].notna()]
    n_correct = int(known['correct'].astype(float).sum()) if len(known) else 0
    acc = (n_correct / len(known)) if len(known) else float('nan')
    rows.append({'comp': cid, 'name': COMP_NAMES.get(cid, ''), 'answered': len(df),
                 'correct': n_correct, 'of': len(known), 'accuracy': acc,
                 # run_reached = THIS sweep's climb (server telemetry); lb_level = all-time scored best.
                 'run_reached': _run_reached(df), 'lb_level': lb_level, 'lb_score': lb_score})
    # Per-competition tool/retrieval usage -- the diagnostic the wrong-dump was MISSING.
    # retrieval_fired = needs_retrieval gated ON; docs_landed = snippets actually came back
    # (fired but 0 docs => the backend returned nothing, e.g. DuckDuckGo blocked on the Colab IP).
    n_retr = int(df['retrieval_used'].fillna(False).astype(bool).sum()) if 'retrieval_used' in df else 0
    n_docs = int(sum(len(_retr_docs(r)) > 0 for _, r in df.iterrows())) if 'retrieved_doc_ids' in df else 0
    n_tool = int(df['tool_used'].notna().sum()) if 'tool_used' in df else 0
    usage.append({'comp': cid, 'name': COMP_NAMES.get(cid, ''), 'answered': len(df),
                  'retrieval_fired': n_retr, 'docs_landed': n_docs, 'tool_calls': n_tool})
    for _, r in df[df['correct'] == False].iterrows():   # correct is None (timeout) -> excluded, good.
        wrong_all.append((cid, COMP_NAMES.get(cid, ''), r))

# --- Scoreboard, per competition + overall ---
# run_reached = how far THIS sweep climbed (server's reached_level telemetry, NOT the always-0 per-turn rung).
# lb_level / lb_score = the AUTHORITATIVE leaderboard numbers (all-time best -- what we are actually scored on).
summary = pd.DataFrame(rows)
print('SCORES BY COMPETITION')
print(summary.to_string(index=False))
tot_c, tot_n = int(summary['correct'].sum()), int(summary['of'].sum())
print((f"\nOVERALL: {tot_c}/{tot_n} = {tot_c / tot_n:.1%}") if tot_n else "\nOVERALL: no graded answers")

# --- RAG / tool usage by competition (the News path, here we finally SEE it) ---
print('\nRAG / TOOL USAGE BY COMPETITION')
print(pd.DataFrame(usage).to_string(index=False))

# --- Every wrong question: text + options + the letter our model picked + WHY (tools/retrieval) ---
print(f"\n{'=' * 72}\nEVERY WRONG QUESTION  ({len(wrong_all)})\n{'=' * 72}")
for cid, name, r in wrong_all:
    opts = r['options'] if 'options' in r and isinstance(r['options'], dict) else {}
    picked = r['predicted_answer']
    docs = _retr_docs(r)
    print(f"\n[comp {cid} · {name}] qid={r['qid']} reached_level={r.get('reached_level')}")
    print(f"Q: {r['question_text']}")
    for letter, text in opts.items():
        mark = '   <-- our pick (WRONG)' if letter == picked else ''
        print(f"   {letter}. {text}{mark}")
    # The diagnostic line -- did a tool fire? did retrieval fire, and did snippets LAND?
    # tool=None + retrieval_used=False  => the plain model answered (no help asked for).
    # retrieval_used=True + docs_landed=0 => the retriever fired but came back EMPTY (the News bug to watch).
    print(f"   -> tool={r.get('tool_used')} | retrieval_used={bool(r.get('retrieval_used'))} | "
          f"docs_landed={len(docs)}" + (f" {docs[:3]}" if docs else ""))

# --- Save a clean consolidated record of the wrong questions (now WITH the tool/retrieval trace) ---
out = os.path.join(REPO_ROOT, 'experiments', 'wrong_questions.jsonl')
with open(out, 'w', encoding='utf-8') as f:
    for cid, name, r in wrong_all:
        f.write(json.dumps({
            'competition_id': cid, 'competition': name, 'qid': r['qid'],
            'level': r['level'], 'reached_level': r.get('reached_level'),
            'question_text': r['question_text'],
            'options': r['options'] if isinstance(r.get('options'), dict) else {},
            'our_wrong_pick': r['predicted_answer'],
            'tool_used': r.get('tool_used'),
            'retrieval_used': bool(r.get('retrieval_used')),
            'retrieved_doc_ids': _retr_docs(r),
        }, ensure_ascii=False) + '\n')
print(f"\nWrong questions saved -> {out}  ({len(wrong_all)} rows)")


SCORES BY COMPETITION
 comp                       name  answered  correct  of  accuracy  run_reached  lb_level  lb_score
    0              Entertainment         6        5   6  0.833333            5        15   1024000
    1 Ancient History & Politics        11       10  11  0.909091           10        15   1024000
    2           Science & Nature         3        2   3  0.666667            2        15   1024000
    3                      Maths         1        0   1  0.000000            0        11     64000
    4    Philosophy & Psychology        15       15  15  1.000000           15        15   1024000
    5                       News         4        3   4  0.750000            3        15   1024000

OVERALL: 35/40 = 87.5%

RAG / TOOL USAGE BY COMPETITION
 comp                       name  answered  retrieval_fired  docs_landed  tool_calls
    0              Entertainment         6                3            3           0
    1 Ancient History & Politics        11               1

In [10]:
import json, collections
from pathlib import Path

# Per-question tool/retrieval/STRATEGY detail across ALL competitions. The diagnostics:
#   - retr=True + docs=0  => the retriever fired but came back EMPTY (DDG blocked / Wikipedia miss).
#   - strat=cot_v1 on comp 3 ONLY  => the Maths routing (pipeline_for) took effect; few_shot_v1 elsewhere.
# Logger is now truncate-per-run (one sweep = one fresh records.jsonl), so these rows are THIS run only.
for cid, path in comp_runs:
    if path is None:
        continue
    p = Path(path) / "records.jsonl"
    if not p.exists():
        continue
    rows = [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]
    print(f"\n===== comp {cid} {COMP_NAMES.get(cid, '')}  (n={len(rows)}) =====")
    print("  tool_used:      ", collections.Counter(r.get("tool_used") for r in rows))
    print("  retrieval_used: ", collections.Counter(r.get("retrieval_used") for r in rows))
    print("  prompt_strategy:", collections.Counter(r.get("prompt_strategy") for r in rows))
    for r in rows:
        docs = r.get("retrieved_doc_ids") or []
        flag = '' if r.get('correct') else '  <-- WRONG'
        print(f"  {r['qid']:>6} strat={str(r.get('prompt_strategy')):<12} "
              f"tool={str(r.get('tool_used')):<11} retr={str(r.get('retrieval_used')):<5} "
              f"docs={len(docs):<2} correct={str(r.get('correct')):<5} | {r['question_text'][:50]}{flag}")



===== comp 0 Entertainment  (n=6) =====
  tool_used:       Counter({None: 6})
  retrieval_used:  Counter({False: 3, True: 3})
  prompt_strategy: Counter({'few_shot_entertainment': 6})
     195 strat=few_shot_entertainment tool=None        retr=False docs=0  correct=True  | What is the primary setting of the film Jurassic P
     392 strat=few_shot_entertainment tool=None        retr=True  docs=3  correct=True  | What is the primary difference between the Beatles
     265 strat=few_shot_entertainment tool=None        retr=True  docs=3  correct=True  | Which of the following best describes the relation
      98 strat=few_shot_entertainment tool=None        retr=False docs=0  correct=True  | What is the fundamental principle behind the forma
     456 strat=few_shot_entertainment tool=None        retr=True  docs=3  correct=True  | What was the main theme of the film 'The Terminato
     365 strat=few_shot_entertainment tool=None        retr=False docs=0  correct=False | What term describes 